# **Import Libraries**

In [202]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# **Import Dataset**

### **Dataset Name :- Online Retail II Data Set from ML Repository**

### *Source :- Kaggle*

### *Link :- https://www.kaggle.com/datasets/mathchi/online-retail-ii-data-set-from-ml-repository/discussion?sort=hotness*

# **Load Datasets**

In [203]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [204]:
file_path = "/content/drive/MyDrive/Project_DA/online_retail_II.xlsx"

In [205]:
path_2009_2010 = "/content/drive/MyDrive/Project_DA/Year 2009-2010.csv"
path_2010_2011 = "/content/drive/MyDrive/Project_DA/Year 2010-2011.csv"

In [206]:
df_2009_2010 = pd.read_csv(path_2009_2010, encoding="ISO-8859-1")
df_2010_2011 = pd.read_csv(path_2010_2011, encoding="ISO-8859-1")

In [207]:
df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)

In [ ]:
df.to_csv("/content/drive/MyDrive/Project_DA/online_retail_raw_combined.csv", index=False)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.columns

# **Data Cleaning**


# 1. Checking for missing values

In [ ]:
df.isnull().sum()

1.1. Dropping rows with missing values or blannk data

In [ ]:
df = df.dropna(subset=['Customer ID'])

1.2. Fill missing values with NULL

In [ ]:
import numpy as np

df['Description'] = df['Description'].replace(r'^\s*$', np.nan, regex=True)

In [ ]:
df.shape

# 2. Checking for Duplicates

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df = df.drop_duplicates(
    subset=['Invoice', 'StockCode', 'InvoiceDate']
)

In [ ]:
df.shape

In [ ]:
df.columns

# 3. Removing Returns and Invalid Transactions


In [ ]:
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

In [ ]:
(df['Quantity'] <= 0).sum(), (df['Price'] <= 0).sum()

# 4. Feature Engineering

4.1 Create Revenue column

In [ ]:
df['Revenue'] = df['Quantity'] * df['Price']

4.2 Converting InvoiceDate to datetime format

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

4.3 Time-Based Features

In [ ]:
df['OrderDate'] = df['InvoiceDate'].dt.date
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)

In [ ]:
df.isnull().sum()

In [ ]:
df[['Quantity', 'Price', 'Revenue']].describe()


In [ ]:
(df['Revenue'] <= 0).sum()

# **Dataset after Cleaning**

In [ ]:
df.to_csv(
    "/content/drive/MyDrive/Project_DA/online_retail_clean.csv",
    index=False
)

In [ ]:
df['InvoiceDate'].min(), df['InvoiceDate'].max()

# **EDA**

In [ ]:
import pandas as pd
df = pd.read_csv(
    "/content/drive/MyDrive/Project_DA/online_retail_clean.csv",
    parse_dates=['InvoiceDate']
)

# 1. Core KPIs

In [ ]:
total_revenue, total_orders, total_customers, aov
kpi_df = pd.DataFrame([{
    "Total Revenue": df['Revenue'].sum(),
    "Total Orders": df['Invoice'].nunique(),
    "Total Customers": df['Customer ID'].nunique(),
    "Average Order Value": df['Revenue'].sum() / df['Invoice'].nunique()
}])

# 2. Revenue Over Time (Monthly)

In [ ]:
df['InvoiceMonth'] = df['InvoiceDate'].dt.to_period('M')

monthly_rev = (
    df.groupby('InvoiceMonth')['Revenue']
      .sum()
      .reset_index()
)

monthly_rev['InvoiceMonth'] = monthly_rev['InvoiceMonth'].astype(str)
monthly_rev.head()

### Line Chart : Plotting the Revenue Trend Plot ( Monthly )

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(monthly_rev['InvoiceMonth'], monthly_rev['Revenue'])
plt.xticks(rotation=45)
plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.show()

# 3. Ranking Top 10 Products ( on the basis of Revenue generated )

In [ ]:
top_products = (
    df.groupby('Description')['Revenue']
      .sum()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()
)

top_products

### Bar Chart : Top Products based on Revenue

In [ ]:
plt.figure(figsize=(10,5))
plt.barh(top_products['Description'], top_products['Revenue'])
plt.gca().invert_yaxis()
plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")
plt.show()

# 4. Revenue by Country (Top 10)

In [ ]:
country_rev = (
    df.groupby('Country')['Revenue']
      .sum()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()
)

country_rev

# 5. Customer Order Frequency

In [ ]:
customer_orders = (
    df.groupby('Customer ID')['Invoice']
      .nunique()
)

customer_orders.describe()

# 6. Repeat vs One-Time Customers

In [ ]:
repeat_customers = (customer_orders > 1).sum()
one_time_customers = (customer_orders == 1).sum()

repeat_customers, one_time_customers

# **Statistical Analysis and Forecasting**

# 1. Monthly Revenue Time Series

In [ ]:
monthly_rev_ts = (
    df.set_index('InvoiceDate')
      .resample('M')['Revenue']
      .sum()
)
monthly_rev_ts.head()

# **Line Chart : Time Series**

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(monthly_rev_ts)
plt.title("Monthly Revenue Time Series")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.show()

# Rolling Average

In [ ]:
rolling_3m = monthly_rev_ts.rolling(window=3).mean()

plt.figure(figsize=(12,5))
plt.plot(monthly_rev_ts, label='Actual')
plt.plot(rolling_3m, label='3-Month Rolling Avg')
plt.legend()
plt.title("Revenue with Rolling Average")
plt.show()

# **Forecasting Model**

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(
    monthly_rev_ts,
    trend='add',
    seasonal=None
)

fit_model = model.fit()
forecast = fit_model.forecast(3)
forecast

# **Forecast**

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(monthly_rev_ts, label='Actual')
plt.plot(forecast, label='Forecast', linestyle='--')
plt.legend()
plt.title("Revenue Forecast (Next 3 Months)")
plt.show()

In [ ]:
forecast_df = forecast.reset_index()
forecast_df.columns = ['Date', 'ForecastedRevenue']

forecast_df.to_csv(
    "/content/drive/MyDrive/Project_DA/revenue_forecast.csv",
    index=False
)

In [ ]:
df.to_csv(
    "/content/drive/MyDrive/Project_DA/online_retail_clean.csv",
    index=False
)

In [ ]:
kpi_df.to_csv(
    "/content/drive/MyDrive/Project_DA/kpi_summary.csv",
    index=False
)

In [ ]:
monthly_revenue = (
    df.groupby('YearMonth')['Revenue']
      .sum()
      .reset_index()
)

monthly_revenue.to_csv(
    "/content/drive/MyDrive/Project_DA/monthly_revenue.csv",
    index=False
)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

monthly_ts = (
    df.set_index('InvoiceDate')
      .resample('M')['Revenue']
      .sum()
)

model = ExponentialSmoothing(monthly_ts, trend='add')
forecast = model.fit().forecast(3)

forecast_df = forecast.reset_index()
forecast_df.columns = ['Date', 'ForecastedRevenue']

forecast_df.to_csv(
    "/content/drive/MyDrive/Project_DA/revenue_forecast.csv",
    index=False
)


In [ ]:
top_products = (
    df.groupby('Description')['Revenue']
      .sum()
      .sort_values(ascending=False)
      .head(10)
      .reset_index()
)

top_products.to_csv(
    "/content/drive/MyDrive/Project_DA/top_products.csv",
    index=False
)

In [ ]:
revenue_by_country = (
    df.groupby('Country')['Revenue']
      .sum()
      .sort_values(ascending=False)
      .reset_index()
)

revenue_by_country.to_csv(
    "/content/drive/MyDrive/Project_DA/revenue_by_country.csv",
    index=False
)